# Session Duration Pattern Analysis

Analyze user session data to understand what influences short vs long sessions, device effects, referrer source impact, and engagement-to-conversion patterns.

## Project Overview

Session duration is a rich engagement signal. Longer sessions often (but not always) signal deeper interest. In this notebook we analyze 4,000 synthetic session records to understand how device type, referrer source, page count, and time of day affect session length — and crucially, how session duration buckets relate to conversion probability.

We also apply a log-transformation to handle the right-skewed nature of duration data, a common real-world challenge.

## Learning Objectives

- Understand why session duration data is typically right-skewed and how to handle it
- Apply log-transformation for visualization and analysis
- Compare session duration across device types, browsers, and referrer sources
- Analyze the relationship between pages viewed and session duration
- Study conversion rates by session duration bucket
- Identify time-of-day patterns in session behavior

## Business or Research Problem

The UX team believes that users from social media referrals have unusually short sessions and low conversion. The engineering team suspects mobile sessions are shorter on average due to poor mobile performance. We need to quantify these hypotheses and identify the session characteristics most predictive of conversion.

## Why This Analysis Matters

Session duration analysis helps prioritize UX investments. If mobile sessions are short and mobile conversion is low, a mobile performance optimization sprint may have higher ROI than a new feature. If social referral sessions are short, the landing page for social traffic needs improvement.

## Dataset Overview

| Column | Type | Description |
|---|---|---|
| session_id | int | Unique session identifier |
| date | date | Session date |
| user_id | int | User identifier |
| device | str | 'desktop', 'mobile', 'tablet' |
| browser | str | Browser used |
| country | str | User country |
| referrer_source | str | Traffic source |
| pages_viewed | int | Pages viewed per session |
| duration_seconds | float | Session duration in seconds |
| converted | int | 1 = purchased |
| hour_of_day | int | Hour session started (0–23) |

4,000 session records.

## Dataset Source and License Notes

Fully synthetic data generated within this notebook using NumPy, Pandas (seed=42). No external dependencies. Educational use only.

## Environment Setup

Standard scientific Python stack: NumPy, Pandas, Matplotlib, Seaborn, SciPy.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('Imports successful.')

## Configuration / Constants

In [ ]:
SEED = 42
N_ROWS = 4000
np.random.seed(SEED)
print(f'Config: {N_ROWS} sessions')

## Dataset Download or Loading

We generate session records. Mobile sessions have a shorter mean duration. Social referrals have shorter mean duration. Duration is lognormally distributed, which is typical for time-on-site data.

In [ ]:
np.random.seed(SEED)

devices = np.random.choice(['desktop', 'mobile', 'tablet'], size=N_ROWS, p=[0.50, 0.40, 0.10])
browsers = np.random.choice(['Chrome', 'Safari', 'Firefox', 'Edge', 'Other'], size=N_ROWS, p=[0.52, 0.25, 0.10, 0.08, 0.05])
countries = np.random.choice(['US', 'UK', 'CA', 'IN', 'AU'], size=N_ROWS, p=[0.45, 0.20, 0.15, 0.12, 0.08])
referrer_source = np.random.choice(['organic', 'paid_search', 'social', 'email', 'direct'], size=N_ROWS, p=[0.30, 0.25, 0.20, 0.15, 0.10])
hour_of_day = np.random.choice(range(24), size=N_ROWS, p=np.array([
    0.01, 0.01, 0.01, 0.01, 0.01, 0.02, 0.03, 0.04,
    0.06, 0.07, 0.07, 0.07, 0.07, 0.07, 0.06, 0.06,
    0.06, 0.06, 0.05, 0.05, 0.04, 0.03, 0.02, 0.01
]))

# Duration depends on device and referrer
duration_mean = np.where(devices == 'mobile', 120,
                np.where(devices == 'tablet', 180, 240))
duration_mean = np.where(referrer_source == 'social', duration_mean * 0.7, duration_mean)
duration_mean = np.where(referrer_source == 'email', duration_mean * 1.2, duration_mean)

duration_seconds = np.random.lognormal(
    mean=np.log(duration_mean),
    sigma=0.8
).clip(10, 3600)

pages_viewed = np.clip(
    (duration_seconds / 60).astype(int) + np.random.randint(1, 4, N_ROWS),
    1, 30
)

# Conversion probability increases with duration up to a point
duration_bucket = np.select(
    [duration_seconds < 60, duration_seconds < 180, duration_seconds < 600, duration_seconds >= 600],
    [0.01, 0.04, 0.10, 0.18]
)
converted = np.random.binomial(1, duration_bucket)

user_ids = np.random.randint(1, 2501, size=N_ROWS)
dates = pd.date_range('2024-01-01', periods=N_ROWS, freq='5min').date
np.random.shuffle(dates)

df = pd.DataFrame({
    'session_id': range(1, N_ROWS + 1),
    'date': dates,
    'user_id': user_ids,
    'device': devices,
    'browser': browsers,
    'country': countries,
    'referrer_source': referrer_source,
    'pages_viewed': pages_viewed,
    'duration_seconds': duration_seconds.round(1),
    'converted': converted,
    'hour_of_day': hour_of_day
})

print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

print(f'\nDuration range: {df["duration_seconds"].min():.1f}s – {df["duration_seconds"].max():.1f}s')
print(f'Median duration: {df["duration_seconds"].median():.1f}s')
print(f'Conversion rate: {df["converted"].mean():.1%}')
print(f'\nDevice distribution:')
print(df['device'].value_counts())

## Data Cleaning

In [ ]:
# Add log-transformed duration
df['log_duration'] = np.log1p(df['duration_seconds'])

# Add duration bucket labels
df['duration_bucket'] = pd.cut(
    df['duration_seconds'],
    bins=[0, 60, 180, 600, 3601],
    labels=['<1 min', '1-3 min', '3-10 min', '10+ min']
)

print('Duration bucket distribution:')
print(df['duration_bucket'].value_counts().sort_index())

## Exploratory Data Analysis

In [ ]:
print('=== Summary Statistics ===')
print(df[['duration_seconds', 'pages_viewed', 'converted']].describe().round(2))

## Univariate Analysis

We compare the raw vs log-transformed duration to demonstrate why the transformation is necessary.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df['duration_seconds'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Raw Session Duration (Right-Skewed)')
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Count')

axes[1].hist(df['log_duration'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Log-Transformed Duration (Approximately Normal)')
axes[1].set_xlabel('log(1 + Duration)')

plt.tight_layout()
plt.show()

**Interpretation:** The raw duration distribution is strongly right-skewed, as expected for time-on-site data. After log-transformation, the distribution is approximately bell-shaped, making it suitable for parametric analysis.

## Bivariate / Multivariate Analysis

Session duration by device and referrer source.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='device', y='log_duration', ax=axes[0], palette='muted')
axes[0].set_title('Log Duration by Device Type')
axes[0].set_ylabel('log(1 + Duration)')

sns.boxplot(data=df, x='referrer_source', y='log_duration', ax=axes[1], palette='muted')
axes[1].set_title('Log Duration by Referrer Source')
axes[1].set_ylabel('log(1 + Duration)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

**Interpretation:** Desktop sessions are longer than mobile and tablet, confirming the hypothesis. Social referral sessions are visibly shorter than email and organic sessions.

## Feature-Specific Insights

Pages viewed vs duration and conversion by bucket.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['pages_viewed'], df['log_duration'], alpha=0.2, color='steelblue', s=10)
axes[0].set_title('Pages Viewed vs Log Duration')
axes[0].set_xlabel('Pages Viewed')
axes[0].set_ylabel('log(1 + Duration)')

cvr_by_bucket = df.groupby('duration_bucket')['converted'].mean()
axes[1].bar(cvr_by_bucket.index.astype(str), cvr_by_bucket.values, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Conversion Rate by Duration Bucket')
axes[1].set_xlabel('Duration Bucket')
axes[1].set_ylabel('Conversion Rate')

plt.tight_layout()
plt.show()

**Interpretation:** There is a positive correlation between pages viewed and session duration. Conversion rate increases dramatically with session duration — sessions over 10 minutes are far more likely to convert than bounce sessions under 1 minute.

## Statistical Checks

In [ ]:
# ANOVA: does device type significantly affect log_duration?
groups = [df[df['device'] == d]['log_duration'] for d in ['desktop', 'mobile', 'tablet']]
f_stat, p_anova = stats.f_oneway(*groups)
print(f'One-way ANOVA (device vs log_duration): F={f_stat:.4f}, p={p_anova:.6f}')
print(f'Result: {"Significant device effect" if p_anova < 0.05 else "No significant effect"}')

# Correlation: pages_viewed vs duration
r, p_corr = stats.pearsonr(df['pages_viewed'], df['log_duration'])
print(f'\nPearson r (pages_viewed vs log_duration): {r:.4f} (p={p_corr:.6f})')

## Visual Analysis

Time-of-day patterns in session volume and duration.

In [ ]:
hourly = df.groupby('hour_of_day').agg(
    sessions=('session_id', 'count'),
    avg_duration=('duration_seconds', 'mean'),
    cvr=('converted', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(hourly['hour_of_day'], hourly['sessions'], color='steelblue', edgecolor='white')
axes[0].set_title('Session Volume by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Session Count')

axes[1].plot(hourly['hour_of_day'], hourly['avg_duration'], marker='o', color='coral')
axes[1].set_title('Average Session Duration by Hour')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Avg Duration (seconds)')

plt.tight_layout()
plt.show()

**Interpretation:** Session volume peaks during business hours (9am–6pm). Average duration tends to be slightly longer in evening hours when users are more relaxed and exploring, compared to quick lunchtime browsing.

## Key Findings

In [ ]:
device_duration = df.groupby('device')['duration_seconds'].median()
referrer_duration = df.groupby('referrer_source')['duration_seconds'].median()

print('=== KEY FINDINGS ===')
print(f'1. Median duration — desktop: {device_duration["desktop"]:.0f}s, mobile: {device_duration["mobile"]:.0f}s')
print(f'2. Median duration — email: {referrer_duration["email"]:.0f}s, social: {referrer_duration["social"]:.0f}s')
print(f'3. Conversion rate <1 min: {cvr_by_bucket["<1 min"]:.1%}')
print(f'4. Conversion rate 10+ min: {cvr_by_bucket["10+ min"]:.1%}')
print(f'5. Pages-duration correlation: r={r:.3f}')
print(f'6. ANOVA p-value (device vs duration): {p_anova:.6f}')

## Limitations

- Duration is modelled as lognormal — real data has bot traffic, infinite scroll, and tab-switching distortions.
- Conversion is modelled as a simple function of duration without controlling for user intent.
- No multi-session user journey is modelled.
- Country differences in session behavior are not modelled.

## Recommendations / Next Steps

1. **Mobile performance audit**: Shorter mobile sessions may reflect slow load times, not lower intent. Run a Core Web Vitals audit.
2. **Social landing page redesign**: Social referral sessions are short — the landing page may not match ad creative expectations.
3. **Session depth targeting**: Users in the 3–10 min bucket are high-intent but not converting — target them with exit-intent offers.
4. **Multi-session analysis**: Combine sessions per user to compute total engagement, not just per-session metrics.
5. **Regression model**: Predict conversion from duration, device, referrer, and pages using logistic regression.

## Common Mistakes

- **Using raw duration for parametric tests**: Always check for skewness and apply log-transform if needed.
- **Including bot sessions**: Bot sessions typically have very short or very long durations and will distort the analysis.
- **Assuming longer = better**: For task-completion sites (e.g., support portals), shorter sessions can mean higher satisfaction.
- **Ignoring referrer source**: Treating all traffic as homogeneous masks important behavioral differences.

## Mini Challenge / Exercises

1. Filter out sessions under 5 seconds (likely bots) and re-compute all statistics.
2. Build a logistic regression model predicting conversion from log_duration, device, and referrer_source.
3. Plot session duration CDFs for each device type on the same chart.
4. Compute the 90th percentile of session duration for each referrer source.
5. Create a heatmap of average session duration by hour_of_day and device.

## Final Summary / Key Takeaways

This notebook analyzed session duration across 4,000 synthetic user sessions:

- Session duration is right-skewed and benefits from log-transformation before analysis.
- Desktop sessions are longer than mobile; social referral sessions are shorter than email.
- Conversion rate increases monotonically with session duration bucket.
- Pages viewed and duration are positively correlated.

**Key principle:** Always investigate *why* sessions are short before concluding engagement is low. Short sessions on mobile may be a performance problem, not a preference.